# Brazilian Tourism Data Pipeline (1989-2024)


## Project Overview
This project demonstrates the end-to-end process of handling real-world historical data. 
It focuses on building a robust ETL (Extract, Transform, Load) pipeline to consolidate 
over 30 years of Brazilian tourism records, addressing challenges like schema drift 
and data inconsistency.


## Table of Contents
0. [Environment Setup](#setup)
1. [Schema Discovery](#schema-discovery)
2. [Data Normalization & Mapping](#normalization)
3. [Incremental Processing Pipeline](#pipeline)
4. [Data Consolidation & Quality Check](#consolidation)
5. [Database Ingestion (SQLite)](#database)

<a id="setup"></a>
## 0. Setup

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

#Global Configuration
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

<a id="schema-discovery"></a>
## 1. Schema Discovery
The first step of our pipeline is to "scan" the raw files. Since this dataset spans 
decades, column names often change. We identify these variations to build a 
mapping strategy without loading the entire dataset into memory.


In [12]:
def get_data_inventory(directory_path: str, separator: str = ';'):
    """
    Scans all CSV files to check if their structure match.
    Returns a list of dictionaries with the different structure among all files.
    """
    path = Path(directory_path)
    files = sorted(list(path.glob("chegadas_*.csv")))
    
    inventory = []
    
    for f in files:
        # Read only the header for efficiency
        header_df = pd.read_csv(f, sep=separator, nrows=0, encoding='latin1')
        cols = header_df.columns.tolist()
        
        inventory.append({
            "filename": f.name,
            "col_count": len(cols),
            "columns": cols
        })
        
    return inventory

# Execute the method
raw_folder = "../data/raw/"
files_inventory = get_data_inventory(raw_folder)

# Create the DataFrame
inventory_df = pd.DataFrame(files_inventory)

# Filter to show only the unique schemas found
pd.set_option('display.max_colwidth', None)
inventory_df['cols_str'] = inventory_df['columns'].astype(str)
unique_schemas = inventory_df.drop_duplicates(subset=['col_count', 'cols_str']).drop(columns=['cols_str'])
print(f"{len(unique_schemas)} different types of file structures have been found.\n")
unique_schemas

3 different types of file structures have been found.



,filename,col_count,columns
0,chegadas_1989.csv,12,"[Continente, Ordem continente, País, Ordem país, UF, Ordem UF, Via de acesso, Ordem via de acesso, ano, Mês, Ordem mês, Chegadas]"
11,chegadas_2000.csv,12,"[Continente, Ordem continente, País, Ordem país, UF, Ordem UF, Via de acesso, Ordem via de acesso, Ano, Mês, Ordem mês, Chegadas]"
27,chegadas_2016.csv,12,"[Continente, cod continente, País, cod pais, UF, cod uf, Via, cod via, ano, Mês, cod mes, Chegadas]"


12 columns were found in every file, which is great. The first two lines returned shows that the only difference is "Ano" being capitalized or lowercase. In the third group, things change a bit more: the column names were shortened, and the capitalization pattern still varies.

<a id="normalization"></a>
## 2. Data Normalization & Mapping

The next step is to standardize everything: decide which names will be our official ones and rename all columns so the data from every year looks exactly the same, also eliminate some of the redundant data found when analyzing only the 1989 file.

In [16]:
def standardize():
    """
    Returns the mapping dictionary and the list of columns to keep.
    This unifies naming patterns and removes redundant data.
    """
    column_mapping = {
        'Continente': 'continent',
        'País': 'country',
        'UF': 'state',
        'Via de acesso': 'arrival_method', 'Via': 'arrival_method',
        'ano': 'year', 'Ano': 'year',
        'Mês': 'month',
        'Chegadas': 'arrivals'
    }
    target_columns = ['continent', 'country', 'state', 'arrival_method', 'year', 'month', 'arrivals']
    
    return column_mapping, target_columns

# Initialize our mapping rules
column_map, target_cols = standardize()

# --- Proof of Concept (Sanity Check) ---
# Testing if the mapping correctly unifies the unique structures from Step 1

def apply_test_mapping(cols_list):
    # Rename and filter in one step for the test
    return [column_map.get(c) for c in cols_list if column_map.get(c) in target_cols]

# Apply the test to our unique_schemas table
unique_schemas['standardized_columns'] = unique_schemas['columns'].apply(apply_test_mapping)

# Display result
print("Verification: If all rows in 'standardized_columns' are identical, the mapping is successful.")
unique_schemas[['filename', 'standardized_columns']]

Verification: If all rows in 'standardized_columns' are identical, the mapping is successful.


,filename,standardized_columns
0,chegadas_1989.csv,"[continent, country, state, arrival_method, year, month, arrivals]"
11,chegadas_2000.csv,"[continent, country, state, arrival_method, year, month, arrivals]"
27,chegadas_2016.csv,"[continent, country, state, arrival_method, year, month, arrivals]"
